# Step 5b L1 Trend Filter — Cell by Cell
---


## 1 Imports and load X_P

In [28]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.tsa.stattools import adfuller, acf
import cvxpy as cp

DATA_DIR = Path("data")

with open(DATA_DIR / "splits.json") as f:
    splits = json.load(f)
train_start, train_end = splits["train"]["start"], splits["train"]["end"]
val_start,   val_end   = splits["val"]["start"],   splits["val"]["end"]
test_start,  test_end  = splits["test"]["start"],  splits["test"]["end"]

factors = pd.read_parquet(DATA_DIR / "spy_factors_step2.parquet")

X_P_COLS = ["CloseLoc", "SignedVolume_z252", "PriceVolCorr_z252", "Overnight",'Mom_60d','Mom_20d']

def signed_log_scaled(x, scale):
    return np.sign(x) * np.log1p(np.abs(x) * scale)

X_P = pd.DataFrame(index=factors.index, columns=X_P_COLS, dtype=float)
X_P["CloseLoc"]            = factors["CloseLoc"]
X_P["SignedVolume_z252"]   = factors["SignedVolume_z252"]
X_P["PriceVolCorr_z252"]   = factors["PriceVolCorr_z252"]
X_P["Overnight"]           = signed_log_scaled(factors["Overnight"], 1e2)
X_P["Mom_60d"]            = factors["Mom_60d"]
X_P["Mom_20d"]            = factors["Mom_20d"]


X_P_train = X_P.loc[train_start:train_end].dropna()
X_P_val   = X_P.loc[val_start:val_end].dropna()
X_P_test  = X_P.loc[test_start:test_end].dropna()

print(f"X_P_train shape: {X_P_train.shape}")
print(f"X_P_val   shape: {X_P_val.shape}")
print(f"X_P_test  shape: {X_P_test.shape}")
print(f"\nX_P_train describe:")
print(X_P_train.describe().loc[["mean", "std", "min", "max"]])

X_P_train shape: (1569, 6)
X_P_val   shape: (336, 6)
X_P_test  shape: (337, 6)

X_P_train describe:
      CloseLoc  SignedVolume_z252  PriceVolCorr_z252  Overnight   Mom_60d  \
mean  0.588984           0.002590           0.012659   0.027409  0.023424   
std   0.318523           1.058760           1.025027   0.442107  0.072521   
min   0.000000          -7.326289          -2.221209  -2.356692 -0.361973   
max   1.000000           5.371131           4.044139   1.848859  0.336620   

       Mom_20d  
mean  0.007601  
std   0.050067  
min  -0.414220  
max   0.207063  


---
## 2 L1 trend filter solver (second-order difference)

In [29]:
def l1_trend_filter(y, lam):
    n = len(y)
    x = cp.Variable(n)
    D2 = np.zeros((n - 2, n))
    for i in range(n - 2):
        D2[i, i]     =  1.0
        D2[i, i + 1] = -2.0
        D2[i, i + 2] =  1.0
    obj = cp.Minimize(0.5 * cp.sum_squares(y - x) + lam * cp.norm1(D2 @ x))
    prob = cp.Problem(obj)
    prob.solve(solver=cp.CLARABEL)
    if x.value is None:
        prob.solve(solver=cp.SCS)
    return np.asarray(x.value).flatten()



---
## 3 λ selection via 5-fold time-series CV (per factor, train segment)

In [30]:
LAMBDA_GRID = np.logspace(-1, 1, 15)
N_SPLITS = 5

def cv_holdout_mse(y, lam, n_splits):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    mses = []
    for tr_idx, te_idx in tscv.split(y):
        if len(tr_idx) < 30 or len(te_idx) < 5:
            continue
        trend_tr = l1_trend_filter(y[tr_idx], lam)
        edge_trend = trend_tr[-1]
        slope = trend_tr[-1] - trend_tr[-2] if len(trend_tr) >= 2 else 0.0
        forecast = edge_trend + slope * np.arange(1, len(te_idx) + 1)
        mses.append(np.mean((y[te_idx] - forecast) ** 2))
    return np.mean(mses) if mses else np.nan

cv_results = {}
for col in X_P_COLS:
    y = X_P_train[col].values
    mses_per_lam = []
    for lam in LAMBDA_GRID:
        mse = cv_holdout_mse(y, lam, N_SPLITS)
        mses_per_lam.append(mse)
    mses_arr = np.array(mses_per_lam)
    best_idx = np.nanargmin(mses_arr)
    cv_results[col] = {
        "lambdas":   LAMBDA_GRID,
        "mses":      mses_arr,
        "best_lam":  LAMBDA_GRID[best_idx],
        "best_mse":  mses_arr[best_idx],
        "at_edge":   (best_idx == 0) or (best_idx == len(LAMBDA_GRID) - 1),
    }

print(f"{'factor':<22} {'best_lam':>10} {'best_mse':>14} {'at_edge?':>10}")
for col in X_P_COLS:
    r = cv_results[col]
    edge = "YES" if r["at_edge"] else "no"
    print(f"{col:<22} {r['best_lam']:>10.4f} {r['best_mse']:>14.6e} {edge:>10}")

print("\nFull CV MSE table (rows: lambda, cols: factor):")
df_cv = pd.DataFrame({col: cv_results[col]["mses"] for col in X_P_COLS},
                     index=[f"{l:.3f}" for l in LAMBDA_GRID])
df_cv.index.name = "lambda"
print(df_cv.to_string(float_format=lambda x: f"{x:.4e}"))

factor                   best_lam       best_mse   at_edge?
CloseLoc                   3.7276   4.106948e-01         no
SignedVolume_z252         10.0000   8.255052e+02        YES
PriceVolCorr_z252         10.0000   1.551110e+01        YES
Overnight                 10.0000   4.301985e-01        YES
Mom_60d                   10.0000   5.374119e-02        YES
Mom_20d                   10.0000   5.243076e-02        YES

Full CV MSE table (rows: lambda, cols: factor):
         CloseLoc  SignedVolume_z252  PriceVolCorr_z252  Overnight    Mom_60d    Mom_20d
lambda                                                                                  
0.100  1.9862e+03         1.3544e+05         6.7906e+03 2.1634e+03 1.0531e-01 2.9630e-01
0.139  8.9477e+02         1.2645e+05         5.4358e+03 1.5919e+03 3.9219e-01 4.8353e-01
0.193  2.9702e+02         1.1381e+05         3.5598e+03 1.8665e+03 6.2626e-01 4.5855e-01
0.268  1.6270e+02         9.6900e+04         2.0673e+03 2.1355e+03 5.6291e-01 4.3842e-

---
## 5 Compute λ_max for each factor (Kim et al. 2009 adaptive scale)

In [31]:
def lambda_max(y):
    n = len(y)
    D2 = np.zeros((n - 2, n))
    for i in range(n - 2):
        D2[i, i]     =  1.0
        D2[i, i + 1] = -2.0
        D2[i, i + 2] =  1.0
    DDT = D2 @ D2.T
    rhs = D2 @ y
    z = np.linalg.solve(DDT, rhs)
    return np.max(np.abs(z))

C_FACTOR = 0.01

lambdas_locked = {}
for col in X_P_COLS:
    y = X_P_train[col].values
    lmax = lambda_max(y)
    lam = C_FACTOR * lmax
    lambdas_locked[col] = {"lambda_max": float(lmax), "lambda": float(lam)}

print(f"{'factor':<22} {'lambda_max':>14} {'lambda (c=0.01)':>18}")
for col in X_P_COLS:
    r = lambdas_locked[col]
    print(f"{col:<22} {r['lambda_max']:>14.4f} {r['lambda']:>18.4f}")

factor                     lambda_max    lambda (c=0.01)
CloseLoc                    1408.2216            14.0822
SignedVolume_z252           2129.9843            21.2998
PriceVolCorr_z252           7532.0036            75.3200
Overnight                   5063.5144            50.6351
Mom_60d                     2435.4833            24.3548
Mom_20d                      807.8871             8.0789


---
## 6 Per-segment L1 decomposition (train / val / test independent)

In [32]:
# ══════════════════════════════════════════════════════════════
# Cell 10 修复版：固定 252 天滚动窗口 L1 滤波（严格因果）
# - Train：整段一次性 L1
# - Val：用 train 末尾 251 天 + val 逐日滚动，窗口固定 252 天
# - Test：用 train+val 末尾 251 天 + test 逐日滚动，窗口固定 252 天
# ══════════════════════════════════════════════════════════════

WINDOW = 252

idx_full      = X_P.dropna().index
trend_full    = pd.DataFrame(index=idx_full, columns=X_P_COLS, dtype=float)
residual_full = pd.DataFrame(index=idx_full, columns=X_P_COLS, dtype=float)

for col in X_P_COLS:
    lam = lambdas_locked[col]["lambda"]
    print(f"\nProcessing {col} (λ={lam:.4f}) ...")

    # ── Train：整段一次性 L1，无泄露问题 ──────────────────────
    y_train = X_P.loc[train_start:train_end, col].dropna()
    trend_train = l1_trend_filter(y_train.values, lam)
    resid_train = y_train.values - trend_train
    trend_full.loc[y_train.index, col]    = trend_train
    residual_full.loc[y_train.index, col] = resid_train
    print(f"  train: {len(y_train)} days, L1 fitted (full segment)")

    # ── Val/Test：固定 252 天滚动窗口 ─────────────────────────
    for seg_name, (s, e) in [("val",  (val_start,  val_end)),
                              ("test", (test_start, test_end))]:
        y_seg = X_P.loc[s:e, col].dropna()
        if len(y_seg) == 0:
            continue

        # 取紧接在当前 segment 之前的 251 天原始数据作为历史前缀
        # val  → 取 train 末尾 251 天
        # test → 取 train+val 末尾 251 天
        if seg_name == "val":
            y_prefix_full = X_P.loc[train_start:train_end, col].dropna()
        else:
            y_prefix_full = X_P.loc[train_start:val_end, col].dropna()

        y_prefix = y_prefix_full.values[-(WINDOW - 1):]   # 取最后 251 天

        trend_seg = np.full(len(y_seg), np.nan)
        for t in range(len(y_seg)):
            # 窗口 = 前缀 251 天 + seg 中 [0, t] 共 t+1 天，取最后 252 天
            y_window = np.concatenate([y_prefix, y_seg.values[:t + 1]])[-WINDOW:]
            try:
                trend_window  = l1_trend_filter(y_window, lam)
                trend_seg[t]  = trend_window[-1]
            except Exception:
                trend_seg[t]  = y_seg.values[t]   # 求解失败时用原始值

        resid_seg = y_seg.values - trend_seg
        trend_full.loc[y_seg.index, col]    = trend_seg
        residual_full.loc[y_seg.index, col] = resid_seg
        print(f"  {seg_name}: {len(y_seg)} days, "
              f"rolling {WINDOW}-day window (prefix={len(y_prefix)} days), "
              f"NaN count={np.isnan(trend_seg).sum()}")

print("\n\nRolling-window L1 decomposition complete.")
print(f"\ntrend_full NaN counts:\n{trend_full.isna().sum()}")
print(f"\nresidual_full NaN counts:\n{residual_full.isna().sum()}")
print(f"\nresidual_full describe (train segment):")
print(residual_full.loc[train_start:train_end].describe().loc[["mean", "std", "min", "max"]])


Processing CloseLoc (λ=14.0822) ...
  train: 1569 days, L1 fitted (full segment)
  val: 336 days, rolling 252-day window (prefix=251 days), NaN count=0
  test: 337 days, rolling 252-day window (prefix=251 days), NaN count=0

Processing SignedVolume_z252 (λ=21.2998) ...
  train: 1569 days, L1 fitted (full segment)
  val: 336 days, rolling 252-day window (prefix=251 days), NaN count=0
  test: 337 days, rolling 252-day window (prefix=251 days), NaN count=0

Processing PriceVolCorr_z252 (λ=75.3200) ...
  train: 1569 days, L1 fitted (full segment)
  val: 336 days, rolling 252-day window (prefix=251 days), NaN count=0
  test: 337 days, rolling 252-day window (prefix=251 days), NaN count=0

Processing Overnight (λ=50.6351) ...
  train: 1569 days, L1 fitted (full segment)
  val: 336 days, rolling 252-day window (prefix=251 days), NaN count=0
  test: 337 days, rolling 252-day window (prefix=251 days), NaN count=0

Processing Mom_60d (λ=24.3548) ...
  train: 1569 days, L1 fitted (full segment)


---
## 9  Save outputs

In [34]:
residual_full.to_parquet(DATA_DIR / "X_P_residual.parquet")
trend_full.to_parquet(DATA_DIR / "X_P_trend.parquet")

with open(DATA_DIR / "l1_lambdas.json", "w") as f:
    json.dump({
        "method": "Kim et al. (2009) lambda_max adaptive scale, c=0.01",
        "C_FACTOR": C_FACTOR,
        "per_factor": lambdas_locked,
        "cv_failure_note": "5-fold TimeSeriesSplit holdout MSE was monotonically decreasing to grid edge for 3/4 factors; "
                           "interpreted as confirmation of weak low-frequency structure in X_P (consistent with Step 4 sparse signal). "
                           "L1 role downgraded from signal enhancement to baseline stationarization; "
                           "L1 vs raw X_P comparison deferred to Step 6.",
    }, f, indent=2)


print("Saved:")
for fn in ["X_P_residual.parquet", "X_P_trend.parquet",
           "l1_lambdas.json"]:
    print(f"  data/{fn}")

Saved:
  data/X_P_residual.parquet
  data/X_P_trend.parquet
  data/l1_lambdas.json
